# InsightAI: Model Validation & Inference EDA
This notebook visualizes the unconstraining of historical demand and validates the LightGBM Quantile predictions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")


In [ ]:
# Load Data
abt = pd.read_parquet('../data/gold/model_input.parquet')
preds = pd.read_csv('../output/insightai_predictions.csv')

# Calculate Historical Max Volume per outlet
historical_max = abt.groupby("Outlet_ID")["Total_Volume"].max().reset_index(name="Historical_Max_Volume")

# Merge
df_plot = preds.merge(historical_max, on="Outlet_ID", how="inner")

# Sort by Historical Max for a clean plot
df_plot = df_plot.sort_values("Historical_Max_Volume").reset_index(drop=True)


In [ ]:
# 1. The "Censored Demand" Scatter Plot
# X-axis: Outlet Index (sorted by size)
# Y-axis: Volume

plt.figure(figsize=(14, 8))

# We'll plot a subset (e.g., top 500 shops) for clarity, or take a random sample
sample_df = df_plot.sample(500, random_state=42).sort_values("Historical_Max_Volume").reset_index(drop=True)

# Plot Historical Actuals (Blue Dots)
plt.scatter(sample_df.index, sample_df["Historical_Max_Volume"], 
            alpha=0.6, color='blue', label='Historical Max Volume (Actuals)')

# Plot Predicted Potential (Green Stars)
plt.scatter(sample_df.index, sample_df["Maximum_Monthly_Liters"], 
            alpha=0.8, color='green', marker='*', s=100, label='Predicted Potential (LightGBM Jan 2026)')

# Highlight the Gap
for i in range(len(sample_df)):
    if sample_df.loc[i, "Maximum_Monthly_Liters"] > sample_df.loc[i, "Historical_Max_Volume"]:
        plt.plot([i, i], 
                 [sample_df.loc[i, "Historical_Max_Volume"], sample_df.loc[i, "Maximum_Monthly_Liters"]], 
                 color='gray', linestyle='--', alpha=0.4)

plt.title("Visualizing Decensoring: Actual vs. Latent Potential Demand (Jan 2026)", fontsize=16, fontweight='bold')
plt.xlabel("Outlet Rank (by Historical Size)", fontsize=12)
plt.ylabel("Volume (Liters)", fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('../output/plot_censored_demand.png', dpi=300)
plt.show()
